# Day 98: Causal Discovery: Learning Graphs from Data

*Beginner to Ninja: Math for Causal Inference*

Everything so far assumed we knew the causal graph. Causal discovery asks how much of the graph the data can reveal. Conditional independencies identify the graph only up to a Markov equivalence class: the skeleton and the colliders. The PC algorithm finds this class with a sequence of independence tests. Extra assumptions, such as non-Gaussian errors in linear models (LiNGAM), can identify directions that independencies cannot.

**Learning outcomes**

- Define Markov equivalence and show which structures are and are not distinguishable from independencies.
- Run the PC algorithm by hand: skeleton search, v-structures and Meek's orientation rule.
- Explain faithfulness and how cancelling paths break it.
- Use non-Gaussianity to orient an edge between two variables.

## Theory

### What independencies reveal

Under the **causal Markov condition** (d-separation implies independence) and **faithfulness** (independence implies d-separation), the set of conditional independencies identifies the graph's **Markov equivalence class**. Two DAGs are Markov equivalent iff they have the same skeleton and the same **v-structures** (colliders $X \to Z \leftarrow Y$ with $X, Y$ non-adjacent) (Verma and Pearl). The class is represented by a CPDAG: directed edges where all members agree, undirected otherwise.

- Chains $X \to Y \to Z$, $X \leftarrow Y \leftarrow Z$ and forks $X \leftarrow Y \to Z$ are equivalent: all imply $X \perp Z \mid Y$.
- The collider $X \to Y \leftarrow Z$ is different: $X \perp Z$ but $X \not\perp Z \mid Y$.

### The PC algorithm

1. **Skeleton.** Start with the complete undirected graph. For $\ell = 0, 1, 2, \dots$: for each adjacent pair $(i, j)$, test $i \perp j \mid S$ for subsets $S$ of size $\ell$ of $i$'s other neighbours; if independent, delete the edge and record the separating set $S_{ij}$.
2. **V-structures.** For each non-adjacent pair $i, j$ with common neighbour $k$: if $k \notin S_{ij}$, orient $i \to k \leftarrow j$.
3. **Meek rules.** Propagate orientations that avoid new v-structures and cycles. Rule 1: if $a \to b - c$ and $a, c$ non-adjacent, orient $b \to c$.

With Gaussian data, test independence with partial correlations and Fisher's $z = \tfrac12\log\frac{1 + r}{1 - r}\sqrt{n - |S| - 3}$.

### Beyond equivalence classes

- **LiNGAM:** in a linear model with independent **non-Gaussian** errors, the full DAG is identifiable: regressing in the wrong direction leaves a residual that is uncorrelated with, but not independent of, the regressor.
- **Additive noise models** extend this to nonlinear relations.
- Background knowledge (time order, "nothing causes age") orients further edges.

### Caveats

Discovery needs causal sufficiency (no hidden common causes; FCI relaxes this), faithfulness and reliable tests; errors in early tests cascade. Treat discovered graphs as hypotheses to be checked with design-based evidence.

## Part 1 · Warm-up

### Exercise 1 · Equivalent or not?

Which of these are Markov equivalent: (a) $A \to B \to C$; (b) $A \leftarrow B \to C$; (c) $A \to B \leftarrow C$?

<details><summary>Show solution</summary>

**Step 1.** (a) and (b) share the skeleton and have no v-structure.

**Answer:** (a) and (b) are equivalent; (c) has a v-structure at $B$ and is not.

</details>

*Your work:*

### Exercise 2 · Fisher's z

A partial correlation $r = 0.05$ is computed with $n = 403$ and $|S| = 0$. Test independence at the 1% level.

<details><summary>Show solution</summary>

**Step 1.** $z = \tfrac12\log(1.05/0.95)\sqrt{400} = 0.05004\times20 = 1.00$.

**Answer:** $p = 0.32$: do not reject independence; the edge would be removed.

</details>

*Your work:*

### Exercise 3 · Orienting a v-structure

PC finds $\text{Diet} - \text{BMI} - \text{Exercise}$ with Diet and Exercise non-adjacent and separated by $S = \{\text{Income}\}$. Orient.

<details><summary>Show solution</summary>

**Step 1.** BMI is not in the separating set.

**Answer:** $\text{Diet} \to \text{BMI} \leftarrow \text{Exercise}$.

</details>

*Your work:*

### Exercise 4 · Meek's rule

After Exercise 3, the graph has $\text{BMI} - \text{BP}$, and BP is adjacent to neither Diet nor Exercise. Orient it.

<details><summary>Show solution</summary>

**Step 1.** $\text{Diet} \to \text{BMI} - \text{BP}$ with Diet and BP non-adjacent.

**Step 2.** $\text{BP} \to \text{BMI}$ would create a new v-structure at BMI, contradicting the independence findings.

**Answer:** $\text{BMI} \to \text{BP}$.

</details>

*Your work:*

### Exercise 5 · Number of tests

Why does PC test small conditioning sets first?

<details><summary>Show solution</summary>

**Answer:** tests with small $S$ are more powerful and cheaper; removing edges early shrinks neighbourhoods so later, larger conditioning sets are drawn from fewer variables.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Chains and forks are equivalent

Show that in linear Gaussian models, $A \to B \to C$ and $A \leftarrow B \to C$ imply the same independencies.

<details><summary>Show solution</summary>

**Step 1.** In both, the only path between $A$ and $C$ passes through $B$ as a non-collider, so $A \perp C \mid B$ by d-separation.

**Step 2.** No other independence holds (all pairs are adjacent or connected through an open path when unconditioned). $\blacksquare$

</details>

*Your work:*

### Exercise 7 · Colliders are detectable

For $A \to B \leftarrow C$ with $B = A + C + \varepsilon$ and independent standard normal $A, C, \varepsilon$, show $\mathrm{corr}(A, C) = 0$ but the partial correlation given $B$ is $-1/2$.

<details><summary>Show solution</summary>

**Step 1.** $A$ and $C$ are independent.

**Step 2.** Residualise on $B$ ($\mathrm{Var}(B) = 3$): $A - B/3$, $C - B/3$.

**Step 3.** $\mathrm{Cov} = 0 - 1/3 - 1/3 + 3/9 = -1/3$; each variance is $1 - 2/3 + 3/9 = 2/3$.

**Step 4.** Partial correlation $= (-1/3)/(2/3) = -1/2$. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · Gaussian pairs are not identifiable

Let $(X, Y)$ be jointly normal with correlation $\rho$, unit variances. Show both $Y = \rho X + e$ and $X = \rho Y + e'$ hold with errors independent of the regressor.

<details><summary>Show solution</summary>

**Step 1.** $e = Y - \rho X$ is normal and uncorrelated with $X$; for jointly normal variables, uncorrelated implies independent.

**Step 2.** The same argument applies to $e' = X - \rho Y$. $\blacksquare$ Both directions are equally valid models.

</details>

*Your work:*

### Exercise 9 · Non-Gaussianity reveals direction

Let $Y = bX + u$ with $X, u$ independent, non-Gaussian, $b \ne 0$. Explain why, in the backward regression $X = cY + r$, the residual $r$ cannot be independent of $Y$.

<details><summary>Show solution</summary>

**Step 1.** $r = (1 - cb)X - cu$ and $Y = bX + u$: both are linear combinations of the independent variables $X$ and $u$.

**Step 2.** $c = b\mathrm{Var}(X)/\mathrm{Var}(Y) \ne 0$ and $1 - cb = \mathrm{Var}(u)/\mathrm{Var}(Y) \ne 0$, so both combinations have non-zero weights on both variables.

**Step 3.** By the Darmois–Skitovich theorem, two such combinations can be independent only if the variables involved are Gaussian. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · A faithfulness violation

Let $X \to Y$ with coefficient $a$ and $X \to M \to Y$ with coefficients $b, c$. Show that if $a + bc = 0$, then $X \perp Y$ despite the causal links.

<details><summary>Show solution</summary>

**Step 1.** $Y = aX + cM + \varepsilon_Y = (a + bc)X + c\varepsilon_M + \varepsilon_Y$.

**Step 2.** With $a + bc = 0$, $Y$ does not depend on $X$ and, in a Gaussian model, is independent of it. $\blacksquare$ PC would wrongly delete the edge $X - Y$.

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · Health behaviours

Data on 3,000 adults are generated from Income $\to$ Diet, Income $\to$ Exercise, Diet $\to$ BMI, Exercise $\to$ BMI, BMI $\to$ blood pressure. Run PC with Fisher-z tests at the 1% level.

<details><summary>Show solution</summary>

**Step 1.** Skeleton: exactly the five true edges (e.g. Diet and Exercise are separated by Income; Income and BP by BMI).

**Step 2.** V-structure: Diet $\to$ BMI $\leftarrow$ Exercise. Meek rule 1: BMI $\to$ BP.

**Step 3.** Income–Diet and Income–Exercise remain undirected: Income could, as far as independencies go, be caused by diet and exercise.

**Answer:** PC recovers the equivalence class; background knowledge (income precedes behaviour) orients the rest.

</details>

*Your work:*

### Exercise 12 · Direction with non-Gaussian data

Weekly alcohol spending $x$ (uniformly distributed) affects a liver-enzyme score $y = 0.8x + u$ with uniform $u$. A dependence measure between regressor and residual is 0.05 for $x \to y$ and 0.54 for $y \to x$; with Gaussian data both are about 0.03–0.05. Interpret.

<details><summary>Show solution</summary>

**Step 1.** In the true direction the residual is independent of the regressor; in the wrong direction it is not (Exercise 9).

**Answer:** non-Gaussianity identifies $x \to y$; with Gaussian data the two directions are indistinguishable.

</details>

*Your work:*

### Exercise 13 · Cancelling pathways

A diuretic lowers blood pressure directly but also triggers a hormonal response that raises it; the two effects roughly cancel. What would PC conclude, and why is that misleading?

<details><summary>Show solution</summary>

**Step 1.** Drug and blood pressure may test as independent, so PC deletes the edge.

**Answer:** faithfulness fails; the drug has two strong effects that offset on average and may not offset in subgroups.

</details>

*Your work:*

### Exercise 14 · Using background knowledge

In a labour-market dataset (age, education, occupation, wage), list two constraints you would impose before running PC.

<details><summary>Show solution</summary>

**Step 1.** Nothing causes age (no arrows into age).

**Step 2.** Time order: education precedes occupation and wage, so arrows from wage to education are forbidden.

</details>

*Your work:*

### Exercise 15 · Communicating discovered graphs

Write two sentences presenting a PC result to a public-health audience responsibly.

<details><summary>Show solution</summary>

**Step 1.** "The data are consistent with diet and exercise each affecting BMI, which affects blood pressure; the data alone cannot tell whether income drives diet or the reverse."

**Step 2.** "These links are hypotheses generated from patterns of association, under the assumption of no unmeasured common causes, and should be tested with designed studies."

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np, itertools as it
from scipy.stats import norm

rng = np.random.default_rng(98)
names = ["Income", "Diet", "Exercise", "BMI", "BP"]; n, p = 3000, 5
I = rng.normal(size=n); Di = .7 * I + rng.normal(size=n); Ex = .6 * I + rng.normal(size=n)
B = .8 * Di - .7 * Ex + rng.normal(size=n); P = .9 * B + rng.normal(size=n)
C = np.corrcoef(np.column_stack([I, Di, Ex, B, P]).T)

def indep(i, j, S, a=.01):                         # Fisher-z test of partial correlation
    idx = [i, j] + list(S); Q = np.linalg.inv(C[np.ix_(idx, idx)]); r = -Q[0, 1] / np.sqrt(Q[0, 0] * Q[1, 1])
    return 2 * norm.sf(abs(np.arctanh(r)) * np.sqrt(n - len(S) - 3)) > a

adj = {i: set(range(p)) - {i} for i in range(p)}; sep = {}
for l in range(p - 1):                             # 1. skeleton
    for i, j in it.permutations(range(p), 2):
        if j not in adj[i]: continue
        for S in it.combinations(sorted(adj[i] - {j}), l):
            if indep(i, j, S):
                adj[i].discard(j); adj[j].discard(i); sep[frozenset((i, j))] = set(S); break
directed = set()
for k in range(p):                                 # 2. v-structures
    for i, j in it.combinations(sorted(adj[k]), 2):
        if j not in adj[i] and k not in sep[frozenset((i, j))]: directed |= {(i, k), (j, k)}
changed = True
while changed:                                     # 3. Meek rule 1
    changed = False
    for a, b in list(directed):
        for c in adj[b]:
            if c != a and c not in adj[a] and (b, c) not in directed and (c, b) not in directed:
                directed.add((b, c)); changed = True
print("directed:  ", sorted((names[a], names[b]) for a, b in directed))
print("undirected:", sorted((names[i], names[j]) for i in range(p) for j in adj[i]
                            if i < j and (i, j) not in directed and (j, i) not in directed))

def dep(a, b):                                     # dependence between regressor and residual
    r = b - np.polyfit(a, b, 1)[0] * a
    return abs(np.corrcoef(a ** 2, r ** 2)[0, 1]) + abs(np.corrcoef(a, r ** 3)[0, 1]) + abs(np.corrcoef(a ** 3, r)[0, 1])
x = rng.uniform(-1, 1, n); y = .8 * x + .6 * rng.uniform(-1, 1, n)
print(f"uniform: x->y {dep(x, y):.3f}   y->x {dep(y, x):.3f}")
xg = rng.normal(size=n); yg = .8 * xg + .6 * rng.normal(size=n)
print(f"gaussian: x->y {dep(xg, yg):.3f}   y->x {dep(yg, xg):.3f}")
print("Ex 2 z:", round(float(np.arctanh(.05) * 20), 3), " p:", round(float(2 * norm.sf(np.arctanh(.05) * 20)), 3))

## Key takeaways

- Independencies identify a graph only up to its Markov equivalence class: the skeleton plus the v-structures.
- PC removes edges with conditional-independence tests, orients colliders, then propagates orientations.
- Non-Gaussian errors (LiNGAM) or background knowledge can orient the remaining edges; faithfulness failures and hidden confounders are the main dangers.